In [1]:
import regex as re
from tqdm import tqdm
import time
from termcolor import colored
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
compiled_pattern = re.compile(GPT4_SPLIT_PATTERN)

In [2]:
def findPairs(text):
    pairs = {}
    i=0
    while(i < len(text)-1):
        pair = (text[i] ,text[i+1])
        pairs[pair] = pairs.get(pair, 0) + 1
        i+=1
    return pairs

def replace(max_pair, ord_text, new_token):
    j=0
    i=0
    temp = []
    while(i < len(ord_text)):
        if(i < len(ord_text)-1 and ord_text[i] == max_pair[0] and ord_text[i+1] == max_pair[1]):
            temp.append(new_token)
            i += 2
            continue
            
        temp.append(ord_text[i])
        i += 1
    return temp

def decode(ids, vocab):
    c=0
    for i in ids:
        s = vocab[i]
        s = s.decode("utf-8", errors="replace")
        color = "on_red" if(c%2) else "on_yellow"
        print(colored(s,"black", color), end="")
        c += 1
        
def encodeV1(text, merges):
    i=0
    tokens = list(text.encode("utf-8"))
    while True:
        i+=1
        pairs = findPairs(tokens)
        index = float("inf")
        to_merge = float("inf")
        for pair, _ in pairs.items():
            cur_index = merges.get(pair, float("inf"))
            if(cur_index < index):
                index = cur_index
                to_merge = pair
        
        if to_merge not in merges:
            break
        tokens = replace(to_merge, tokens, index)
    return tokens

def encodeV2(text, merges):
    words = re.findall(compiled_pattern, text)
    parts = []
    for text in words:
        tokens = list(text.encode("utf-8"))
        while True:
            pairs = findPairs(tokens)
            index = float("inf")
            to_merge = float("inf")
            for pair, _ in pairs.items():
                cur_index = merges.get(pair, float("inf"))
                if(cur_index < index):
                    index = cur_index
                    to_merge = pair
            
            if to_merge not in merges:
                parts.extend(tokens)
                break
            tokens = replace(to_merge, tokens, index)
    return parts

def display(message, merges, vocab):
    # tokensV1 = encodeV1(message, merges)
    tokensV2 = encodeV2(message, merges)
    # decode(tokensV1, vocab)
    # print()
    decode(tokensV2, vocab)
    print()

In [3]:
import json

with open("encoder.json", "r") as f:
    vocab = json.load(f)
    
with open("vocab.bpe", "r", encoding="utf-8") as f:
    bpe_data = f.read()

merges = [tuple(merge_str.split()) for merge_str in bpe_data.split("\n")[1:-1]]

In [4]:
display("hello world", merges, vocab)

AttributeError: 'list' object has no attribute 'get'

In [15]:
file_count = 0
with open(f"Dataset/en_{file_count}.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
uniqe_words = set(re.findall(compiled_pattern, raw_text[:500000]))
print("Total Uniqe Words: ", len(uniqe_words))

Total Uniqe Words:  15928


In [4]:
print(len(raw_text))
sentences = raw_text.split(".")
print(len(sentences))
batch = "".join(sentences[10:20])
print(batch)

48460658
460569
 Once children progress to secondary school (9th grade), they attend boarding school and return home during school breaks Through education, these children will have the resources necessary to provide a better future for themselves and for their families—one where they are never faced with the decision of whether or not to abandon their own children Maternal Health Right now, 1 in 18 women in Malawi will die during their lifetime due to complications and the lack of medical care during pregnancy and delivery For every 100,000 births that occur in Malawi this year, more than 800 mothers will die during delivery or shortly after because of complications The correlation between high rates of maternal mortality and the number of orphaned children in a country is strong, and Malawi estimates that there are already more than one million orphans 100X Development is working with several key government, nonprofit, and academic partners to develop the Live to Love program Live to

In [5]:
test = "Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'—words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!"

In [9]:
vocab = {idx: bytes([idx]) for idx in range (256)}

In [10]:
def train(raw_text, num_merges = 512):
    merges = {}
    sentences = raw_text.split(".")
    batch_size = 10000
    k = 0
    for j in tqdm(range(0, len(sentences), batch_size)):
        raw_batch = "".join(sentences[j:j+batch_size])
        raw_bytes = raw_batch.encode("utf-8")
        for i in (range(num_merges)):
            # Find Max Pair
            pairs = findPairs(raw_bytes)
            max_pair = max(pairs, key = lambda x: pairs[x])
            if(pairs[max_pair] == 1):
                    break
            # Mint New Token
            new_token = 256 + k
            k += 1
            merges[max_pair] = new_token
            # Merge
            raw_bytes = replace(max_pair, raw_bytes, new_token)
            
        # Print once in a while
        
        for (p1, p2), idx in merges.items():
            vocab[idx] = vocab[p1] + vocab[p2]
        display(test, merges, vocab)
        time.sleep(30)
    return merges

In [70]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
compiled_pattern = re.compile(GPT4_SPLIT_PATTERN)

In [8]:
def getstat(text, pairs):
    i=0
    while(i < len(text)-1):
        pair = (text[i] ,text[i+1])
        pairs[pair] = pairs.get(pair, 0) + 1
        i+=1

In [18]:
vocab2 = {idx: bytes([idx]) for idx in range (256)}
def train2(raw_text, num_merges = 15000):
    merges = {}
    text_chunks = re.findall(compiled_pattern, raw_text)
    ids = [list(ch.encode("utf-8")) for ch in text_chunks]
    
    for i in tqdm((range(num_merges))):
        # Find Max Pair
        pairs = {}
        for chunk_ids in ids:
            getstat(chunk_ids, pairs)
            
        max_pair = max(pairs, key = lambda x: pairs[x])
        if(pairs[max_pair] == 1):
                break
        # Mint New Token
        new_token = 256 + i
        merges[max_pair] = new_token
        # Merge
        ids = [replace(max_pair, chunk_ids, new_token) for chunk_ids in ids]

    # Print once in a while
    for (p1, p2), idx in merges.items():
        vocab2[idx] = vocab2[p1] + vocab2[p2]
    display(test, merges, vocab2)
        # return merges
    return merges

In [19]:
merges2 = train2(raw_text[:800000])

100%|██████████| 15000/15000 [55:23<00:00,  4.51it/s] 

Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'—words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!


In [29]:
lol = """
50 of the people that were eliminated in this video were randomly selected to come back for episode 1 of Beast Games due to scheduling conflicts and other contestants unable to attend.
All footage was reviewed! for? all eliminations, decisions and any actions taken by contestants.
Thank you to the Las, Vegas Raiders & Allegiant Stadium for hosting us at their incredible stadium. 

"""

In [30]:
display(lol, merges2, vocab2)


50 of the people that were eliminated in this video were randomly selected to come back for episode 1 of Beast Games due to scheduling conflicts and other contestants unable to attend.
All footage was reviewed! for? all eliminations, decisions and any actions taken by contestants.
Thank you to the Las, Vegas Raiders & Allegiant Stadium for hosting us at their incredible stadium. 




In [32]:
import pickle

with open("vocab2.pkl", "wb") as f:
    pickle.dump(vocab2, f)

In [33]:
with open("merges2.pkl", "wb") as f:
    pickle.dump(merges2, f)

In [20]:
len(vocab2)

15256

In [11]:
merges = train(raw_text)

  0%|          | 0/47 [00:00<?, ?it/s]

Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!
Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!


  2%|▏         | 1/47 [01:41<1:17:36, 101.23s/it]

Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!
Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!


  4%|▍         | 2/47 [02:59<1:05:54, 87.88s/it] 

...

 96%|█████████▌| 45/47 [1:12:16<03:16, 98.41s/it]

Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!
Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!


 98%|█████████▊| 46/47 [1:13:54<01:38, 98.21s/it]

Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!
Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'��words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!


100%|██████████| 47/47 [1:14:28<00:00, 95.08s/it]


In [12]:
len(merges)

16370

In [27]:
display(msg, merges, vocab)


Once the vocabulary is built, tokenize new text by applying the learned merge rules iteratively until no further merges can be applied.


Once the vocabulary is built, tokenize new text by applying the learned merge rules iteratively until no further merges can be applied.



In [40]:
len(vocab)

24320